# Sanity Check - Step 04: ICA Artifact Removal

Überprüft:
- ICA Komponenten identifiziert
- EOG Artefakte erkannt
- Vorher/Nachher Signalqualität
- Amplituden-Reduktion durch ICA

In [ ]:
import sys
import mne
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / 'eeg_pipeline'))
import config

print("Setup erfolgreich")

## 1. VORHER: Gefilterte Raw Daten (Step 03 Output)

In [ ]:
subject_id = config.SUBJECTS[0]

# Step 03 Output laden
p1_path_before = config.OUTPUT_DIR / f"sub-{subject_id}_P1_filtered.fif"
p2_path_before = config.OUTPUT_DIR / f"sub-{subject_id}_P2_filtered.fif"

raw_p1_before = mne.io.read_raw_fif(str(p1_path_before), preload=False)
raw_p2_before = mne.io.read_raw_fif(str(p2_path_before), preload=False)

print(f"\n=== VORHER (Step 03 Output - Gefiltert) ===\n")
print(f"Person 1:")
print(f"  EEG Kanäle: {len(mne.pick_types(raw_p1_before.info, eeg=True))}")
print(f"  EOG Kanäle: {len(mne.pick_types(raw_p1_before.info, eog=True))}")
print(f"  Sampling Rate: {raw_p1_before.info['sfreq']} Hz")
print(f"  Datenlänge: {raw_p1_before.n_times} Samples ({raw_p1_before.times[-1]:.2f}s)")

print(f"\nPerson 2:")
print(f"  EEG Kanäle: {len(mne.pick_types(raw_p2_before.info, eeg=True))}")
print(f"  EOG Kanäle: {len(mne.pick_types(raw_p2_before.info, eog=True))}")
print(f"  Sampling Rate: {raw_p2_before.info['sfreq']} Hz")
print(f"  Datenlänge: {raw_p2_before.n_times} Samples ({raw_p2_before.times[-1]:.2f}s)")

## 2. NACHHER: ICA Cleaned Raw Daten (Step 04 Output)

In [ ]:
# Step 04 Output laden
p1_path_after = config.OUTPUT_DIR / f"sub-{subject_id}_P1_ica_cleaned.fif"
p2_path_after = config.OUTPUT_DIR / f"sub-{subject_id}_P2_ica_cleaned.fif"

raw_p1_after = mne.io.read_raw_fif(str(p1_path_after), preload=False)
raw_p2_after = mne.io.read_raw_fif(str(p2_path_after), preload=False)

print(f"\n=== NACHHER (Step 04 Output - ICA Cleaned) ===\n")
print(f"Person 1:")
print(f"  EEG Kanäle: {len(mne.pick_types(raw_p1_after.info, eeg=True))}")
print(f"  EOG Kanäle: {len(mne.pick_types(raw_p1_after.info, eog=True))}")
print(f"  Sampling Rate: {raw_p1_after.info['sfreq']} Hz")
print(f"  Datenlänge: {raw_p1_after.n_times} Samples ({raw_p1_after.times[-1]:.2f}s)")

print(f"\nPerson 2:")
print(f"  EEG Kanäle: {len(mne.pick_types(raw_p2_after.info, eeg=True))}")
print(f"  EOG Kanäle: {len(mne.pick_types(raw_p2_after.info, eog=True))}")
print(f"  Sampling Rate: {raw_p2_after.info['sfreq']} Hz")
print(f"  Datenlänge: {raw_p2_after.n_times} Samples ({raw_p2_after.times[-1]:.2f}s)")

## 3. Amplituden Statistiken Vergleich

In [ ]:
# Hole EEG Daten
eeg_picks = mne.pick_types(raw_p1_before.info, eeg=True)
raw_p1_before.load_data()
raw_p1_after.load_data()
data_before = raw_p1_before.get_data(picks=eeg_picks)
data_after = raw_p1_after.get_data(picks=eeg_picks)

eog_picks = mne.pick_types(raw_p1_before.info, eog=True)
if len(eog_picks) > 0:
    eog_data_before = raw_p1_before.get_data(picks=eog_picks)
    eog_data_after = raw_p1_after.get_data(picks=eog_picks)
else:
    eog_data_before = None
    eog_data_after = None

print(f"\n=== AMPLITUDEN STATISTIKEN (Person 1) ===\n")

print(f"EEG DATEN:")
print(f"\n  VORHER (gefiltert):")
print(f"    Min: {np.min(data_before):.6f} µV")
print(f"    Max: {np.max(data_before):.6f} µV")
print(f"    Mean: {np.mean(data_before):.6f} µV")
print(f"    Std: {np.std(data_before):.6f} µV")
print(f"    Peak-to-Peak: {np.max(data_before) - np.min(data_before):.6f} µV")

print(f"\n  NACHHER (ICA cleaned):")
print(f"    Min: {np.min(data_after):.6f} µV")
print(f"    Max: {np.max(data_after):.6f} µV")
print(f"    Mean: {np.mean(data_after):.6f} µV")
print(f"    Std: {np.std(data_after):.6f} µV")
print(f"    Peak-to-Peak: {np.max(data_after) - np.min(data_after):.6f} µV")

print(f"\n  ÄNDERUNG:")
print(f"    Amplitude-Reduktion (Std): {(1 - np.std(data_after)/np.std(data_before))*100:.2f}%")
print(f"    Peak-to-Peak-Reduktion: {(1 - (np.max(data_after) - np.min(data_after))/(np.max(data_before) - np.min(data_before)))*100:.2f}%")

if eog_data_before is not None:
    print(f"\nEOG DATEN (Augenbewegungen):")
    print(f"\n  VORHER:")
    print(f"    Std: {np.std(eog_data_before):.6f} µV")
    
    print(f"\n  NACHHER (nach ICA):")
    print(f"    Std: {np.std(eog_data_after):.6f} µV")
    
    print(f"\n  ÄNDERUNG:")
    print(f"    EOG Amplitude-Reduktion: {(1 - np.std(eog_data_after)/np.std(eog_data_before))*100:.2f}%")

## 4. PSD Vergleich: VORHER vs. NACHHER

In [ ]:
# PSD plotten
fig = plt.figure(figsize=(14, 6))

# VORHER
plt.subplot(1, 2, 1)
raw_p1_before_eeg = raw_p1_before.copy().pick_types(eeg=True).load_data()
raw_p1_before_eeg.plot_psd(fmax=50, ax=plt.gca(), show=False)
plt.title('VORHER: Power Spectral Density')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power (µV²/Hz)')

# NACHHER
plt.subplot(1, 2, 2)
raw_p1_after_eeg = raw_p1_after.copy().pick_types(eeg=True).load_data()
raw_p1_after_eeg.plot_psd(fmax=50, ax=plt.gca(), show=False)
plt.title('NACHHER: Power Spectral Density (nach ICA)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power (µV²/Hz)')

plt.tight_layout()
plt.show()

## 5. Zeitbereich Vergleich: Ein EEG Kanal

In [ ]:
# Vergleiche ein Kanal im Zeitbereich
eeg_picks = mne.pick_types(raw_p1_before.info, eeg=True)
ch_names = mne.pick_types(raw_p1_before.info, eeg=True, ret_names=True)[0]

# Wähle einen exemplarischen EEG Kanal (zentral)
ch_idx = 15  # Mittlerer Kanal
ch_name = ch_names[ch_idx]

# Extraiere Zeitreihe (erste 10 Sekunden)
t_end = min(10, raw_p1_before.times[-1])
t_idx_end = int(t_end * raw_p1_before.info['sfreq'])

data_before_ch = raw_p1_before.get_data(picks=eeg_picks)[ch_idx, :t_idx_end]
data_after_ch = raw_p1_after.get_data(picks=eeg_picks)[ch_idx, :t_idx_end]
times = raw_p1_before.times[:t_idx_end]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# VORHER
ax1 = axes[0]
ax1.plot(times, data_before_ch, 'b-', linewidth=0.5, label='Ungereinigte')
ax1.set_ylabel('Amplitude (µV)')
ax1.set_title(f'VORHER: Kanal {ch_name} - Ungereinigte Daten (erste {t_end:.1f}s)')
ax1.grid(True, alpha=0.3)
ax1.legend()

# NACHHER
ax2 = axes[1]
ax2.plot(times, data_after_ch, 'g-', linewidth=0.5, label='ICA Gereinigt')
ax2.set_xlabel('Zeit (Sekunden)')
ax2.set_ylabel('Amplitude (µV)')
ax2.set_title(f'NACHHER: Kanal {ch_name} - Nach ICA Artefakt-Entfernung (erste {t_end:.1f}s)')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Vergleichter Kanal: {ch_name}")

## 6. Artefakt-Verteilung nach Kanälen

In [ ]:
# Berechne relative Amplitude-Reduktion pro Kanal (korreliert mit Artefakten)
data_before_all = raw_p1_before.get_data(picks=eeg_picks)
data_after_all = raw_p1_after.get_data(picks=eeg_picks)

std_before = np.std(data_before_all, axis=1)
std_after = np.std(data_after_all, axis=1)
std_reduction = ((std_before - std_after) / std_before) * 100

fig, axes = plt.subplots(1, 1, figsize=(14, 6))

x = np.arange(len(ch_names))
colors = ['green' if r > 0 else 'red' for r in std_reduction]

axes.bar(x, std_reduction, color=colors, alpha=0.7)
axes.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes.set_xlabel('Kanal Index')
axes.set_ylabel('Amplitude-Reduktion (%)')
axes.set_title('Artefakt-Reduktion pro Kanal nach ICA')
axes.grid(True, alpha=0.3, axis='y')
axes.set_xticks(x[::4])
axes.set_xticklabels(ch_names[::4], rotation=45)

plt.tight_layout()
plt.show()

print(f"\nDurchschnittliche Amplitude-Reduktion: {np.mean(std_reduction):.2f}%")
print(f"Range: {np.min(std_reduction):.2f}% bis {np.max(std_reduction):.2f}%")
print(f"Kanäle mit Artefakten (Reduktion > 10%): {np.sum(std_reduction > 10)}")

## 7. EOG (Augenbewegungen) Vergleich

In [ ]:
# Vergleiche EOG Signal vor und nach ICA
if eog_picks:
    eog_names = mne.pick_types(raw_p1_before.info, eog=True, ret_names=True)[0]
    
    fig, axes = plt.subplots(len(eog_names), 1, figsize=(14, 4*len(eog_names)))
    
    if len(eog_names) == 1:
        axes = [axes]
    
    # Verwende Zeitfenster
    t_end = min(20, raw_p1_before.times[-1])
    t_idx_end = int(t_end * raw_p1_before.info['sfreq'])
    times = raw_p1_before.times[:t_idx_end]
    
    for i, (eog_pick, eog_name) in enumerate(zip(eog_picks, eog_names)):
        eog_before = raw_p1_before.get_data(picks=[eog_pick])[0, :t_idx_end]
        eog_after = raw_p1_after.get_data(picks=[eog_pick])[0, :t_idx_end]
        
        ax = axes[i]
        ax.plot(times, eog_before, 'b-', linewidth=0.5, label='VORHER', alpha=0.7)
        ax.plot(times, eog_after, 'g-', linewidth=0.5, label='NACHHER (ICA gereinigt)', alpha=0.7)
        ax.set_ylabel(f'{eog_name} (µV)')
        ax.set_title(f'{eog_name} - Vorher/Nachher Vergleich (erste {t_end:.1f}s)')
        ax.grid(True, alpha=0.3)
        ax.legend()
    
    axes[-1].set_xlabel('Zeit (Sekunden)')
    plt.tight_layout()
    plt.show()
else:
    print("Keine EOG Kanäle vorhanden.")

In [ ]:
# Zusammenfassung für Person 2
eeg_picks_p2 = mne.pick_types(raw_p2_before.info, eeg=True)
data_p2_before = raw_p2_before.get_data(picks=eeg_picks_p2)
data_p2_after = raw_p2_after.get_data(picks=eeg_picks_p2)

print(f"\n=== PERSON 2 STATISTIKEN ===\n")

print(f"EEG Amplitude-Reduktion (Std): {(1 - np.std(data_p2_after)/np.std(data_p2_before))*100:.2f}%")
print(f"EEG Peak-to-Peak-Reduktion: {(1 - (np.max(data_p2_after) - np.min(data_p2_after))/(np.max(data_p2_before) - np.min(data_p2_before)))*100:.2f}%")

## 8. Zusammenfassung & Validierung

In [ ]:
print(f"\n=== SANITY CHECK ZUSAMMENFASSUNG ===\n")

checks = []

# 1. Daten-Shape sollte gleich sein
same_shape = data_before.shape == data_after.shape
checks.append(("Daten-Shape gleich", same_shape))

# 2. Sampling Rate sollte gleich sein
same_sfreq = raw_p1_before.info['sfreq'] == raw_p1_after.info['sfreq']
checks.append(("Sampling Rate gleich", same_sfreq))

# 3. Kanal-Namen sollten gleich sein
same_channels = raw_p1_before.ch_names == raw_p1_after.ch_names
checks.append(("Kanal-Namen gleich", same_channels))

# 4. Amplitude sollte durch ICA reduziert sein
amp_reduced = np.std(data_after) < np.std(data_before)
checks.append(("EEG Amplitude reduziert (ICA entfernte Artefakte)", amp_reduced))

# 5. Reduktion sollte vernünftig sein (nicht zu viel)
reduction_reasonable = (1 - np.std(data_after)/np.std(data_before)) < 0.5  # Weniger als 50%
checks.append(("Amplitude-Reduktion vernünftig (<50%)", reduction_reasonable))

# 6. Keine NaN oder Inf
no_nan_inf = not (np.isnan(data_after).any() or np.isinf(data_after).any())
checks.append(("Keine NaN/Inf Werte", no_nan_inf))

# 7. Zeitdauer sollte gleich sein
same_duration = abs(raw_p1_before.times[-1] - raw_p1_after.times[-1]) < 0.01
checks.append(("Zeitdauer gleich", same_duration))

# 8. ICA sollte auch Person 2 verarbeitet haben
p2_processed = np.std(data_p2_after) < np.std(data_p2_before)
checks.append(("Person 2 auch verarbeitet", p2_processed))

for check_name, result in checks:
    status = "✓ PASS" if result else "✗ FAIL"
    print(f"{status}: {check_name}")

all_pass = all(result for _, result in checks)
print(f"\n{'='*50}")
if all_pass:
    print("✓ ALLE CHECKS BESTANDEN")
else:
    print("✗ EINIGE CHECKS FEHLGESCHLAGEN")
print(f"{'='*50}")